In [ ]:
# ── CELL 0: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)
print('Ready.')

In [ ]:
# ── CELL 1: Traditional Programming vs Machine Learning ───────────────────────
# TRADITIONAL: human writes rules by hand
def is_spam_traditional(email):
    if 'FREE' in email.upper():      return True
    if 'CLICK HERE' in email.upper(): return True
    if 'WINNER' in email.upper():    return True
    return False

test_emails = [
    'Congratulations! You are a WINNER! Click here for FREE prize!',
    'Hi, can we reschedule tomorrow meeting?',
    'URGENT: Claim your FREE lottery winnings NOW',
    'Your package has been shipped. Track it here.',
]

print('Traditional Rule-Based:')
for e in test_emails:
    print(f"  {'SPAM' if is_spam_traditional(e) else 'HAM ':4s} | {e[:55]}")

# ML: algorithm discovers rules from examples
train_emails = [
    'FREE money click here now winner',
    'Congratulations you won FREE prize claim now',
    'URGENT FREE gift claim your winnings today',
    'Buy cheap pills online FREE delivery',
    'Hi can we meet tomorrow for coffee',
    'The quarterly report is attached for review',
    'Let me know when you are available to talk',
    'Your invoice for last month is ready',
    'Team lunch is at noon today see you there',
    'Please review the pull request when you get a chance',
]
train_labels = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

spam_model = Pipeline([('vec', CountVectorizer()), ('clf', MultinomialNB())])
spam_model.fit(train_emails, train_labels)

print('\nML-Based (learned from 10 examples):')
for e in test_emails:
    pred = spam_model.predict([e])[0]
    prob = spam_model.predict_proba([e])[0][1]
    print(f"  {'SPAM' if pred else 'HAM ':4s} ({prob*100:.0f}% spam) | {e[:50]}")

In [ ]:
# ── CELL 2: The Learning Process Visualized ────────────────────────────────────
np.random.seed(42)
n = 50
experience = np.sort(np.random.randint(1, 30, n)).astype(float)
salary     = 40000 + experience * 3000 + np.random.randn(n) * 8000

w, b  = 100.0, 1000.0   # bad random starting values
lr    = 0.01
history = {'loss': [], 'w': []}

print(f"{'Step':>5} | {'Weight':>10} | {'Loss (MSE)':>15}")
for step in range(500):
    yp  = w * experience + b
    mse = np.mean((salary - yp) ** 2)
    history['loss'].append(mse); history['w'].append(w)
    err = yp - salary
    w  -= lr * np.mean(err * experience)
    b  -= lr * np.mean(err)
    if step in [0, 50, 200, 499]:
        print(f"{step:>5} | {w:>10.1f} | {mse:>15,.0f}")

print(f"\nTrue:    salary = 3000 * exp + 40000")
print(f"Learned: salary = {w:.0f} * exp + {b:.0f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(experience, salary, alpha=0.6, color='steelblue')
xl = np.array([experience.min(), experience.max()])
axes[0].plot(xl, w*xl+b, 'tomato', lw=2, label=f'Learned: y={w:.0f}x+{b:.0f}')
axes[0].set_title('Learned Function'); axes[0].legend()
axes[1].plot(history['loss'], 'steelblue', lw=2)
axes[1].set_title('Loss Curve — Model Getting Better'); axes[1].set_yscale('log')
axes[2].plot(history['w'], 'seagreen', lw=2, label='learned weight')
axes[2].axhline(3000, color='tomato', ls='--', label='true slope=3000')
axes[2].set_title('Weight Converging to True Value'); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 3: ML Vocabulary — Features, Labels, Model, Training, Inference ───────
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'age':        np.random.randint(22, 60, n),
    'experience': np.random.randint(0, 35, n),
    'score':      np.random.normal(75, 10, n).clip(40, 100),
    'dept':       np.random.choice([1, 2, 3, 4], n),
})
df['salary'] = (30000 + df['experience']*2800 + df['age']*500
                + df['score']*200 + df['dept']*3000
                + np.random.randn(n)*8000)

# FEATURES (X) — the inputs
X = df[['age','experience','score','dept']].values
y = df['salary'].values
print(f'Features X: {X.shape}  (rows=samples, cols=features)')
print(f'Target y:   {y.shape}  (one salary per sample)')

# TRAIN/TEST SPLIT — test set is sacred, never used during training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

# TRAINING — model learns the mapping X -> y
model = LinearRegression()
model.fit(X_train, y_train)
print(f'\nLearned coefficients: {model.coef_.round(0)}')
for name, coef in zip(['age','experience','score','dept'], model.coef_):
    print(f'  +1 {name:12s} -> +${coef:,.0f} salary')

# INFERENCE — predict on new, never-seen data
new_emp = np.array([[30,7,85,1],[45,20,72,3],[28,3,92,4]])
preds   = model.predict(new_emp)
print('\nInference on new employees:')
for emp, pred in zip(new_emp, preds):
    print(f'  age={emp[0]} exp={emp[1]} score={emp[2]} dept={emp[3]} -> ${pred:,.0f}')

# EVALUATION — how well does it generalize?
rmse_tr = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
rmse_te = np.sqrt(mean_squared_error(y_test,  model.predict(X_test)))
r2      = r2_score(y_test, model.predict(X_test))
print(f'\nTrain RMSE: ${rmse_tr:,.0f}  |  Test RMSE: ${rmse_te:,.0f}  |  R²: {r2:.4f}')

In [ ]:
# ── CELL 4: The Full ML Project Skeleton (7 Steps) ────────────────────────────
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'sqft':        np.random.normal(1800, 500, n).clip(600, 5000).round(),
    'bedrooms':    np.random.randint(1, 6, n),
    'bathrooms':   np.random.randint(1, 4, n),
    'age':         np.random.randint(0, 60, n),
    'dist_center': np.random.exponential(10, n).clip(0.5, 50).round(1),
})
df['price'] = (100000 + df['sqft']*120 + df['bedrooms']*15000
               + df['bathrooms']*20000 - df['age']*1500
               - df['dist_center']*3000 + np.random.randn(n)*30000).clip(50000)
df.loc[df.sample(frac=0.05,random_state=1).index,'sqft']  = np.nan
df.loc[df.sample(frac=0.03,random_state=2).index,'price'] = np.nan
df = pd.concat([df, df.sample(10,random_state=3)], ignore_index=True)

# Step 1: Load
print(f'Step 1 — Loaded: {df.shape}')

# Step 2: EDA
print(f'Step 2 — Missing: {df.isnull().sum()[df.isnull().sum()>0].to_dict()}')
print(f'         Duplicates: {df.duplicated().sum()}')

# Step 3: Clean
df = df.drop_duplicates().reset_index(drop=True)
df['sqft'] = df['sqft'].fillna(df['sqft'].median())
df = df.dropna(subset=['price']).reset_index(drop=True)
print(f'Step 3 — After cleaning: {df.shape}')

# Step 4: Features
feat_cols = ['sqft','bedrooms','bathrooms','age','dist_center']
X = df[feat_cols].values; y = df['price'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)
print(f'Step 4 — Train: {X_tr_sc.shape}  Test: {X_te_sc.shape}')

# Step 5: Train
model = LinearRegression()
model.fit(X_tr_sc, y_train)
print(f'Step 5 — Model trained: {model.__class__.__name__}')

# Step 6: Evaluate
rmse_tr = np.sqrt(mean_squared_error(y_train, model.predict(X_tr_sc)))
rmse_te = np.sqrt(mean_squared_error(y_test,  model.predict(X_te_sc)))
r2      = r2_score(y_test, model.predict(X_te_sc))
print(f'Step 6 — RMSE train: ${rmse_tr:,.0f}  test: ${rmse_te:,.0f}  R²: {r2:.4f}')

# Step 7: Inference
new_h = np.array([[2000,3,2,10,5],[1200,2,1,40,15],[3500,5,3,2,25]])
preds = model.predict(scaler.transform(new_h))
print('Step 7 — Predictions:')
for i, (h, p) in enumerate(zip(new_h, preds)):
    print(f'  House {i+1}: {h[0]:.0f}sqft {h[1]}bed {h[3]}yr {h[4]}km -> ${p:,.0f}')

fig, ax = plt.subplots(figsize=(7,6))
yp = model.predict(X_te_sc)
ax.scatter(y_test/1000, yp/1000, alpha=0.4, s=20, color='steelblue')
lims = [min(y_test.min(), yp.min())/1000, max(y_test.max(), yp.max())/1000]
ax.plot(lims, lims, 'r--', lw=2)
ax.set_title(f'Predicted vs Actual Price  R²={r2:.3f}')
ax.set_xlabel('Actual ($K)'); ax.set_ylabel('Predicted ($K)')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 5: Overfitting and Underfitting Visualized ────────────────────────────
np.random.seed(42)
X_all = np.sort(np.random.uniform(0, 2*np.pi, 40))
y_all = np.sin(X_all) + np.random.randn(40) * 0.3
X_tr, X_te = X_all[:30], X_all[30:]
y_tr, y_te = y_all[:30], y_all[30:]
X_plot = np.linspace(0, 2*np.pi, 300)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
configs = [
    (1,  'UNDERFITTING (deg=1)\nToo simple',       'tomato'),
    (4,  'GOOD FIT (deg=4)\nGeneralizes well',     'seagreen'),
    (20, 'OVERFITTING (deg=20)\nMemorizes noise',  'orange'),
]
for ax, (deg, title, color) in zip(axes, configs):
    m = Pipeline([('p', PolynomialFeatures(deg)), ('r', LinearRegression())])
    m.fit(X_tr.reshape(-1,1), y_tr)
    rtr = np.sqrt(mean_squared_error(y_tr, m.predict(X_tr.reshape(-1,1))))
    rte = np.sqrt(mean_squared_error(y_te, m.predict(X_te.reshape(-1,1))))
    ax.scatter(X_tr, y_tr, color='steelblue', s=30, alpha=0.7, label='Train')
    ax.scatter(X_te, y_te, color='tomato',    s=50, alpha=0.9, marker='*', label='Test')
    ax.plot(X_plot, np.sin(X_plot), 'k--', alpha=0.4, lw=1.5, label='True')
    ax.plot(X_plot, m.predict(X_plot.reshape(-1,1)), color=color, lw=2.5, label='Model')
    ax.set_title(f'{title}\nTrain RMSE={rtr:.3f}  Test RMSE={rte:.3f}')
    ax.legend(fontsize=8); ax.set_ylim(-2.5, 2.5)
plt.suptitle('Underfitting | Good Fit | Overfitting', fontsize=13)
plt.tight_layout(); plt.show()

print('Train RMSE >> Test threshold: neither too simple nor too complex')
print('Large train-test gap -> overfitting -> regularize or simplify')
print('Both RMSEs high -> underfitting -> add features or complexity')

In [ ]:
# ── CELL 6: Practice — Dubai Marina Rental Prediction ─────────────────────────
np.random.seed(42)
n = 400
df = pd.DataFrame({
    'size_sqft':  np.random.normal(1200, 400, n).clip(400, 4000).round(),
    'bedrooms':   np.random.choice([0,1,2,3,4], n, p=[0.10,0.25,0.35,0.20,0.10]),
    'floor':      np.random.randint(1, 50, n),
    'age_years':  np.random.randint(0, 20, n),
    'sea_view':   np.random.choice([0,1], n, p=[0.6,0.4]),
    'parking':    np.random.choice([0,1,2], n, p=[0.3,0.5,0.2]),
})
df['rent_aed'] = (30000 + df['size_sqft']*60 + df['bedrooms']*20000
                  + df['floor']*500 - df['age_years']*1000
                  + df['sea_view']*35000 + df['parking']*8000
                  + np.random.randn(n)*20000).clip(25000)
df.loc[df.sample(frac=0.06,random_state=1).index,'size_sqft'] = np.nan
df = pd.concat([df, df.sample(8,random_state=2)], ignore_index=True)

# EDA
print(f'Shape: {df.shape}  Missing: {df.isnull().sum()[df.isnull().sum()>0].to_dict()}')

# Clean
df = df.drop_duplicates().reset_index(drop=True)
df['size_sqft'] = df['size_sqft'].fillna(df['size_sqft'].median())

# Prepare
feat_cols = ['size_sqft','bedrooms','floor','age_years','sea_view','parking']
X = df[feat_cols].values; y = df['rent_aed'].values
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# Train
model = LinearRegression()
model.fit(X_tr_sc, y_train)

# Evaluate
rmse_tr = np.sqrt(mean_squared_error(y_train, model.predict(X_tr_sc)))
rmse_te = np.sqrt(mean_squared_error(y_test,  model.predict(X_te_sc)))
r2      = r2_score(y_test, model.predict(X_te_sc))
print(f'Train RMSE: AED {rmse_tr:,.0f}  |  Test RMSE: AED {rmse_te:,.0f}  |  R²: {r2:.4f}')

print('\nFeature impact (per std dev):')
for name, coef in sorted(zip(feat_cols, model.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {"+" if coef>0 else "-"} {name:12s}: AED {abs(coef):>8,.0f}/yr')

# Predict 3 apartments
apts = np.array([[900,1,5,15,0,1],[1800,2,25,5,1,1],[3000,4,45,1,1,2]])
preds = model.predict(scaler.transform(apts))
labels = ['Studio, low floor, old','2BR, mid floor, sea view','4BR penthouse']
print('\nRent predictions:')
for lbl, pred in zip(labels, preds):
    print(f'  {lbl}: AED {pred:,.0f}/yr  ({pred/12:,.0f}/mo)')

gap = (rmse_te - rmse_tr) / rmse_tr * 100
print(f'\nFit: gap={gap:.1f}%  R²={r2:.3f}  -> {"good generalization" if gap<20 and r2>0.7 else "check for issues"}')

fig, ax = plt.subplots(figsize=(7,6))
yp = model.predict(X_te_sc)
ax.scatter(y_test/1000, yp/1000, alpha=0.5, s=20, color='steelblue')
lims=[y_test.min()/1000,y_test.max()/1000]
ax.plot(lims,lims,'r--',lw=2)
ax.set_title(f'Dubai Marina — Predicted vs Actual Rent  R²={r2:.3f}')
ax.set_xlabel('Actual (AED 000s)'); ax.set_ylabel('Predicted (AED 000s)')
plt.tight_layout(); plt.show()